# Fine-tuning BERT on GLUE SST-2

This clean notebook documents the 15,000-example baseline as an executable workflow. The original Colab notebook remains preserved at the repository root. This notebook has not been run during the audit, so its metric values will only exist after you execute it.

The official SST-2 test labels are hidden. This notebook selects checkpoints and reports metrics on validation only.


## 1. Imports and environment
PyTorch runs training; Transformers provides BERT and Trainer; Datasets loads GLUE; scikit-learn calculates classification metrics.

In [1]:
import random
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, Trainer, TrainingArguments, set_seed

SEED = 42
MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 128
TRAIN_SIZE = 15_000
OUTPUT_DIR = Path("models/best_model")


c:\Users\mittu\Documents\GitHub-Projects\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Reproducibility and compute
Set seeds before sampling or model initialization. Deterministic kernels can reduce speed and do not guarantee bitwise identity across hardware or library versions.

In [2]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
set_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


## 3. Dataset loading and inspection
SST-2 supplies labeled training and validation splits. The test labels are hidden (`-1`), so they cannot support a standard local test report. Checkpoint selection uses validation only.

In [4]:
dataset = load_dataset("nyu-mll/glue", "sst2")
train_full = dataset["train"]
validation = dataset["validation"]
print(dataset)
print("Columns:", train_full.column_names)
print("Example:", train_full[0])
print("Validation rows:", len(validation))


c:\Users\mittu\Documents\GitHub-Projects\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mittu\.cache\huggingface\hub\datasets--nyu-mll--glue. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\mittu\Documents\GitHub-Projects\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: User

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})
Columns: ['sentence', 'label', 'idx']
Example: {'sentence': 'hide new secretions from the parental units ', 'label': 0, 'idx': 0}
Validation rows: 872


In [ ]:
from collections import Counter
print("Train labels:", Counter(train_full["label"]))
print("Validation labels:", Counter(validation["label"]))
for row in train_full.select(range(3)):
    print(row["label"], row["sentence"])


## 4. Preprocessing and tokenization
We retain text and labels. BERT's uncased WordPiece tokenizer maps words/subwords to **input IDs** (vocabulary indices). **Attention masks** mark content tokens (1) versus padding (0). Truncation limits sequences to 128 tokens. Dynamic padding pads each batch only to its longest sequence.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def tokenize_batch(batch):
    return tokenizer(batch["sentence"], truncation=True, max_length=MAX_LENGTH)

# First 15,000 official training rows match the documented baseline subset.
small_train = train_full.select(range(TRAIN_SIZE))
tokenized_train = small_train.map(tokenize_batch, batched=True)
tokenized_validation = validation.map(tokenize_batch, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print(tokenizer("A pleasant surprise.", truncation=True, max_length=MAX_LENGTH))


## 5. BERT and the classification objective
BERT is a Transformer encoder pretrained with bidirectional context. Self-attention lets each token representation incorporate information from other tokens. Fine-tuning updates pretrained weights for this task. The `[CLS]` representation feeds a two-logit classification head. Cross-entropy compares logits with the gold class; AdamW optimizes the model using warmup and a linear learning-rate schedule.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2,
    id2label={0: "Negative", 1: "Positive"},
    label2id={"Negative": 0, "Positive": 1},
)


## 6. Training configuration
This preserves 15k examples, batch size 32, two epochs, learning rate `2e-5`, and weight decay `0.01`. Epoch evaluation and checkpointing are aligned. The best checkpoint is selected by validation F1; gradient clipping limits large updates. GPU is recommended; CPU works but a complete run may be slow.

In [ ]:
training_args = TrainingArguments(
    output_dir="results/baseline_15k",
    eval_strategy="epoch", save_strategy="epoch",
    logging_strategy="steps", logging_steps=50,
    learning_rate=2e-5, per_device_train_batch_size=32,
    per_device_eval_batch_size=32, num_train_epochs=2,
    weight_decay=0.01, warmup_ratio=0.1,
    lr_scheduler_type="linear", max_grad_norm=1.0,
    load_best_model_at_end=True, metric_for_best_model="eval_f1",
    greater_is_better=True, save_total_limit=2,
    seed=SEED, data_seed=SEED, report_to="none",
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="binary", zero_division=0
    )
    return {"accuracy": accuracy_score(labels, predictions),
            "precision": precision, "recall": recall, "f1": f1}

trainer = Trainer(model=model, args=training_args,
    train_dataset=tokenized_train, eval_dataset=tokenized_validation,
    data_collator=data_collator, compute_metrics=compute_metrics)


## 7. Fine-tune
This is the expensive training step. It downloads BERT weights if needed, then runs two epochs. Nothing starts automatically from inference or tests.

In [ ]:
train_result = trainer.train()
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Training summary:", train_result.metrics)


## 8. Validation evaluation
These are validation measurements, not test results. Accuracy is the fraction correct. Positive-class precision, recall, and F1 summarize positive predictions. The confusion matrix uses true labels as rows and predicted labels as columns.

In [ ]:
validation_output = trainer.predict(tokenized_validation, metric_key_prefix="validation")
y_true = np.asarray(validation["label"])
y_pred = np.argmax(validation_output.predictions, axis=-1)
print("Validation loss:", validation_output.metrics.get("validation_loss"))
print("Validation metrics:", {k:v for k,v in validation_output.metrics.items() if k.startswith("validation_")})
print("Confusion matrix [true rows, predicted columns]:\n", confusion_matrix(y_true, y_pred, labels=[0,1]))
print(classification_report(y_true, y_pred, labels=[0,1], target_names=["Negative","Positive"], zero_division=0))


## 9. Training and validation loss curves
The plot uses actual Trainer logs from the current run; it does not contain placeholder results.

In [ ]:
history = trainer.state.log_history
train_points = [(x.get("step"), x["loss"]) for x in history if "loss" in x and "eval_loss" not in x]
eval_points = [(x.get("step"), x["eval_loss"]) for x in history if "eval_loss" in x]
plt.figure(figsize=(8,4))
if train_points: plt.plot(*zip(*train_points), label="Training loss", alpha=.8)
if eval_points: plt.plot(*zip(*eval_points), marker="o", label="Validation loss")
plt.xlabel("Training step"); plt.ylabel("Cross-entropy loss")
plt.title("Baseline training and validation loss"); plt.legend(); plt.grid(alpha=.2); plt.tight_layout(); plt.show()


## 10. Prediction examples and error analysis
Inspect representative mistakes for ambiguity, negation, sarcasm, or label noise. Qualitative validation errors do not justify tuning on the hidden test set.

In [ ]:
examples = ["A delightful, funny film with a moving ending.", "A dull and painfully predictable movie."]
encoded = tokenizer(examples, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
encoded = {key:value.to(trainer.args.device) for key,value in encoded.items()}
model.eval()
with torch.no_grad():
    probs = torch.softmax(model(**encoded).logits, dim=-1).cpu().numpy()
for text, p in zip(examples, probs):
    label = model.config.id2label[int(np.argmax(p))]
    print({"text":text, "label":label, "probabilities":p.tolist()})


In [ ]:
import pandas as pd
validation_frame = pd.DataFrame({"sentence":validation["sentence"], "true_label":y_true, "predicted_label":y_pred})
errors = validation_frame[validation_frame.true_label != validation_frame.predicted_label]
print(f"Misclassified validation examples: {len(errors)} / {len(validation_frame)}")
errors.head(12)


## 11. Save the model and tokenizer
Save the best validation-selected checkpoint locally. It is ignored by Git and required by `src/predict.py`.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print("Saved model and tokenizer to", OUTPUT_DIR.resolve())


## 12. Test split note
GLUE SST-2 test labels are hidden. The test split can be used to generate benchmark submissions, but it cannot produce local classification metrics. This workflow reports training and validation results only.